# AHD e HD arteriais no ImageCAS

O notebook mede as distâncias entre a artéria predita e a referência com
`hausdorff_distances` (SimpleITK). Ambas as máscaras são avaliadas na mesma
grade, usando todos os voxels positivos e o espaçamento em milímetros.

- **HD (distância de Hausdorff).** Em cada sentido, mede-se a distância do centro
  de cada voxel positivo ao centro do voxel positivo mais próximo na outra
  máscara. A HD é a maior dessas distâncias nos dois sentidos.
- **AHD (distância média de Hausdorff, ou AvgHD).** É a média aritmética das duas
  médias direcionais. Cada sentido recebe o mesmo peso.

Configure `IMAGECAS_BASE_PATH` se o dataset estiver fora dos caminhos padrões
do projeto.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.ticker import MaxNLocator

cwd = Path.cwd().resolve()
for candidate in (cwd, *cwd.parents):
    if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
        src_dir = str(candidate / "src")
        if src_dir not in sys.path:
            sys.path.insert(0, src_dir)
        break
else:
    raise FileNotFoundError("Abra o notebook a partir de uma pasta do repositório.")

from utils.processing.gpu_utils import use_gpu
from utils.project.analysis.artery_distance_diagnostics import (
    sample_foreground_distances,
)
from utils.project.results.io import results_filename
from utils.project.runtime.notebook_env import (
    configure_notebook_environment,
    load_notebook_pipeline_config,
    resolve_imagecas_base_path,
)
from utils.segmentation.pipeline.arteries import segment_artery_masks_from_vesselness
from utils.segmentation.pipeline.detection import detect_ostia, locate_and_segment_aorta
from utils.segmentation.pipeline.preprocessing import (
    compute_vesselness,
    load_and_preprocess_image,
)
from utils.utils.metrics import dice_score, hausdorff_distances
from utils.utils.roi import mask_voxel_bbox_slices
from utils.visualization.images.volume import (
    visualize_binary_masks_comparison,
    visualize_label_map_3d,
)

REPO_ROOT = configure_notebook_environment(chdir_to_src=False)

## Exemplos de exames

Os resultados vêm do canônico SciPy `mid_res/val`, cuja média de Dice é
60,61% em 270 exames. A tabela usa as medidas de **superfície** para HD e
AHD e lista os exames em ordem decrescente de Dice.

In [ ]:
example_cases = pd.DataFrame(
    [
        (899, 36.53, 1.120, 79.53),
        (873, 28.21, 0.878, 79.37),
        (730, 22.71, 0.819, 77.95),
        (935, 10.18, 0.606, 76.49),
        (605, 19.93, 0.841, 74.58),
        (790, 17.26, 0.754, 73.22),
        (920, 28.98, 1.752, 67.56),
        (981, 41.05, 2.719, 63.62),
        (869, 25.70, 2.307, 60.44),
        (437, 38.64, 3.232, 58.45),
        (307, 84.56, 21.103, 32.26),
        (162, 49.53, 6.855, 31.39),
    ],
    columns=["Exame", "HD (mm)", "AHD (mm)", "Dice (%)"],
).sort_values("Dice (%)", ascending=False)
display(example_cases.set_index("Exame"))

## Configuração

O limiar em milímetros define as faixas de distância nas figuras. Para consultar
apenas a distribuição no canônico, execute os imports, a configuração e a
célula do histograma canônico.

In [ ]:
IMG_ID = 869
RESOLUTION = "mid"  # "mid" ou "high"
HIGH_DISTANCE_MM = 10.0
HISTOGRAM_BINS = 40

CANONICAL_RESOLUTION = f"{RESOLUTION}_res"  # "mid_res" ou "high_res"
CANONICAL_SPLIT = "test"  # "train", "val" ou "test"

CONFIG_FILE = REPO_ROOT / "config" / "pipeline_config.json"
USE_GPU = None  # None respeita a configuração do pipeline.

PREDICTION_COLOR = "#2F80ED"
REFERENCE_COLOR = "#39B54A"

if not np.isfinite(HIGH_DISTANCE_MM) or HIGH_DISTANCE_MM < 0:
    raise ValueError("HIGH_DISTANCE_MM deve ser um valor finito e não negativo.")
if not isinstance(HISTOGRAM_BINS, int) or HISTOGRAM_BINS < 1:
    raise ValueError("HISTOGRAM_BINS deve ser um inteiro positivo.")
if CANONICAL_RESOLUTION not in {"mid_res", "high_res"}:
    raise ValueError("CANONICAL_RESOLUTION deve ser 'mid_res' ou 'high_res'.")
if CANONICAL_SPLIT not in {"train", "val", "test"}:
    raise ValueError("CANONICAL_SPLIT deve ser 'train', 'val' ou 'test'.")

## Predição da artéria

A localização da aorta e dos óstios e a segmentação arterial usam a configuração
do pipeline. A comparação usa a predição após o pós-processamento morfológico
e o rótulo coronariano (`label == 1`) na mesma grade processada.

In [ ]:
config = load_notebook_pipeline_config(CONFIG_FILE, RESOLUTION)
gpu_requested = config.get("USE_GPU", False) if USE_GPU is None else USE_GPU
config["USE_GPU"] = bool(gpu_requested and use_gpu())
print(
    f"Exame {IMG_ID} | Resolução {RESOLUTION} | "
    f"Processamento {'GPU' if config['USE_GPU'] else 'CPU'}"
)

imagecas_base_path = resolve_imagecas_base_path()
image_data = load_and_preprocess_image(str(IMG_ID), str(imagecas_base_path), config)
lcc_image = image_data["lcc_image"]
reference_mask = image_data["label"] == 1
# Usa o espaçamento efetivo na ordem dos eixos da grade, como no canonical.
spacing_mm = image_data["spacing"]
scaled_spacing_mm = image_data["scaled_spacing"]
downscale_factors = tuple(image_data["downscale_factors"])
aorta_stage = locate_and_segment_aorta(
    lcc_image,
    downscale_factors,
    scaled_spacing_mm,
    config,
    require_non_empty=True,
)
del image_data

print("Espaçamento Original (mm):", tuple(spacing_mm))
print("Espaçamento Scaled (mm):", tuple(scaled_spacing_mm))
print(
    "Tamanho Fisico Scaled (mm):",
    tuple((np.array(lcc_image.shape) * scaled_spacing_mm).tolist()),
)

In [ ]:
vesselness_ostia = compute_vesselness(
    lcc_image, config["VESSELNESS_AORTA"], use_gpu=config["USE_GPU"]
)
ostium_left, ostium_right = detect_ostia(
    aorta_stage.aorta_mask, vesselness_ostia, scaled_spacing_mm, config
)
del aorta_stage, vesselness_ostia

vesselness_artery = compute_vesselness(
    lcc_image, config["VESSELNESS_ARTERY"], use_gpu=config["USE_GPU"]
)
artery = segment_artery_masks_from_vesselness(
    lcc_image, vesselness_artery, ostium_left, ostium_right, config
)
prediction_mask = artery.final_mask > 0
print(f"Método arterial: {artery.method}")
del artery, lcc_image, vesselness_artery

## Métricas arteriais

In [ ]:
hausdorff = (
    hausdorff_distances(prediction_mask, reference_mask, scaled_spacing_mm)
    if prediction_mask.any() and reference_mask.any()
    else None
)
hd_mm, ahd_mm = hausdorff or (None, None)
display(
    pd.DataFrame(
        [
            ("Dice", f"{dice_score(prediction_mask, reference_mask):.2%}"),
            ("HD (mm)", f"{hd_mm:.6f}" if hd_mm is not None else "Indisponível"),
            ("AHD (mm)", f"{ahd_mm:.6f}" if ahd_mm is not None else "Indisponível"),
        ],
        columns=["Métrica", "Resultado"],
    ).set_index("Métrica")
)
if hausdorff is None:
    print("HD e AHD indisponíveis: pelo menos uma das máscaras está vazia.")

## Predição e referência em 3D

A cena mostra a predição em azul e a referência em verde. As malhas podem ser
alternadas pelo menu da cena.

In [ ]:
def plot_artery_masks(prediction, reference, spacing, title):
    """Exibe as malhas e preserva as coordenadas físicas após o recorte."""
    print(
        f"{title} | Predição: {np.count_nonzero(prediction):,} voxels | "
        f"Referência: {np.count_nonzero(reference):,} voxels"
    )
    bbox = mask_voxel_bbox_slices(prediction | reference)
    if bbox is None:
        print("Nenhum voxel para exibir.")
        return None

    # Uma borda de fundo fecha a malha mesmo quando o vaso toca a borda da grade.
    pred_crop = np.pad(prediction[bbox], 1)
    ref_crop = np.pad(reference[bbox], 1)
    pred_color = int(PREDICTION_COLOR[1:], 16)
    ref_color = int(REFERENCE_COLOR[1:], 16)
    if pred_crop.any() and ref_crop.any():
        plot = visualize_binary_masks_comparison(
            ref_crop,
            pred_crop,
            spacing=spacing,
            display_plot=False,
            plot_name=title,
            reference_name="Artéria GT",
            predicted_name="Artéria predita",
            reference_color=ref_color,
            predicted_color=pred_color,
            reference_opacity=0.34,
            predicted_opacity=0.44,
        )
    else:
        mask, name, color = (
            (pred_crop, "Artéria predita", pred_color)
            if pred_crop.any()
            else (ref_crop, "Artéria GT", ref_color)
        )
        plot = visualize_label_map_3d(
            mask,
            {1: (name, color)},
            spacing=spacing,
            plot_name=title,
            display_plot=False,
        )
    offset_mm = (np.array([axis.start for axis in bbox]) - 1) * spacing
    for mesh in plot.objects:
        mesh.vertices = (mesh.vertices + offset_mm).astype(np.float32)
    plot.axes = ["X (mm)", "Y (mm)", "Z (mm)"]
    display(plot)
    return plot

In [ ]:
full_plot = plot_artery_masks(
    prediction_mask,
    reference_mask,
    scaled_spacing_mm,
    f"Exame {IMG_ID} · Predição e referência",
)

## Distribuição das distâncias no exame

Cada observação corresponde a um voxel positivo da máscara de origem.
Os painéis usam os mesmos intervalos e limites de eixo nos dois sentidos.
A linha tracejada marca o limiar usado nas cenas de distância.

In [ ]:
directional_distances = sample_foreground_distances(
    prediction_mask, reference_mask, scaled_spacing_mm
)
pred_to_ref_mm, ref_to_pred_mm = directional_distances or (None, None)

if pred_to_ref_mm is None or ref_to_pred_mm is None:
    print("Histograma indisponível: pelo menos uma das máscaras está vazia.")
else:
    distance_max = max(pred_to_ref_mm.max(), ref_to_pred_mm.max())
    edges = np.linspace(0, distance_max or 1.0, HISTOGRAM_BINS + 1)
    fig, axes = plt.subplots(
        1, 2, figsize=(14, 5), sharex=True, sharey=True, layout="constrained"
    )
    fig.suptitle(f"Exame {IMG_ID} · Distâncias entre máscaras")
    for ax, values, title, color in (
        (axes[0], pred_to_ref_mm, "Predição → referência", PREDICTION_COLOR),
        (axes[1], ref_to_pred_mm, "Referência → predição", REFERENCE_COLOR),
    ):
        ax.hist(values, bins=edges, color=color, alpha=0.8, edgecolor="black")
        ax.axvline(
            HIGH_DISTANCE_MM,
            color="#C62828",
            linestyle="--",
            label=f"Limiar de {HIGH_DISTANCE_MM:g} mm",
        )
        ax.set(title=title, xlabel="Distância ao vizinho mais próximo (mm)")
        ax.grid(axis="y", alpha=0.3)
        ax.set_axisbelow(True)
        ax.legend()
    axes[0].set_ylabel("Voxels positivos")
    plt.show()

## Distribuição de AHD e HD no run canônico

Cada observação representa um exame com HD e AHD válidas da máscara inteira.
Os runs que registram esse cálculo usam as colunas
`artery_hd_foreground_mm` e `artery_avghd_foreground_mm`.

In [ ]:
canonical_csv = (
    REPO_ROOT
    / "output"
    / "segmentation"
    / "canonical"
    / CANONICAL_RESOLUTION
    / CANONICAL_SPLIT
    / "current"
    / "numeric"
    / results_filename(CANONICAL_SPLIT)
)
print(f"Fonte: {canonical_csv.relative_to(REPO_ROOT)}")

foreground_columns = ("artery_avghd_foreground_mm", "artery_hd_foreground_mm")
status_column = "artery_distance_metrics_status"
if not canonical_csv.is_file():
    print("Resultados canônicos indisponíveis para a seleção.")
else:
    canonical_results = pd.read_csv(canonical_csv)
    if not {status_column, *foreground_columns}.issubset(canonical_results.columns):
        print("O CSV selecionado não contém HD e AHD da máscara inteira.")
    else:
        canonical_cases = canonical_results.loc[
            canonical_results[status_column].eq("ok"), list(foreground_columns)
        ].apply(pd.to_numeric, errors="coerce")
        ahd = canonical_cases["artery_avghd_foreground_mm"]
        hd = canonical_cases["artery_hd_foreground_mm"]
        canonical_cases = canonical_cases.loc[
            np.isfinite(ahd) & np.isfinite(hd) & ahd.ge(0) & hd.ge(ahd)
        ]
        print(
            f"Exames com AHD e HD da máscara inteira válidas: "
            f"{len(canonical_cases)}/{len(canonical_results)}"
        )
        if not canonical_cases.empty:
            fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
            fig.suptitle(
                f"ImageCAS · Canonical {CANONICAL_RESOLUTION}/{CANONICAL_SPLIT}"
            )
            for ax, metric, column, color in zip(
                axes,
                ("AHD", "HD"),
                foreground_columns,
                ("skyblue", "lightgreen"),
                strict=True,
            ):
                ax.hist(
                    canonical_cases[column],
                    bins=HISTOGRAM_BINS,
                    color=color,
                    alpha=0.8,
                    edgecolor="black",
                )
                ax.set(title=metric, xlabel="Distância (mm)", ylabel="Exames")
                ax.grid(axis="y", alpha=0.3)
                ax.set_axisbelow(True)
                ax.yaxis.set_major_locator(MaxNLocator(integer=True))
            plt.show()

## Regiões com distâncias altas e baixas em 3D

As cenas de distância **alta** (≥ `HIGH_DISTANCE_MM`) mostram os voxels
positivos de origem acima do limiar junto à máscara de destino completa.
A cena de distância **baixa** mostra os voxels abaixo do limiar nas duas
máscaras.

### Distâncias altas da predição para a referência

A cena sobrepõe os voxels positivos da predição com distância alta à
referência completa.

In [ ]:
if pred_to_ref_mm is None:
    print("Distâncias indisponíveis: pelo menos uma máscara está vazia.")
else:
    high_prediction = np.zeros_like(prediction_mask)
    high_prediction[prediction_mask] = pred_to_ref_mm >= HIGH_DISTANCE_MM
    high_prediction_plot = plot_artery_masks(
        high_prediction,
        reference_mask,
        scaled_spacing_mm,
        f"Predição → referência · ≥ {HIGH_DISTANCE_MM:g} mm",
    )

### Distâncias altas da referência para a predição

A cena sobrepõe os voxels positivos da referência com distância alta à
predição completa.

In [ ]:
if ref_to_pred_mm is None:
    print("Distâncias indisponíveis: pelo menos uma máscara está vazia.")
else:
    high_reference = np.zeros_like(reference_mask)
    high_reference[reference_mask] = ref_to_pred_mm >= HIGH_DISTANCE_MM
    high_reference_plot = plot_artery_masks(
        prediction_mask,
        high_reference,
        scaled_spacing_mm,
        f"Referência → predição · ≥ {HIGH_DISTANCE_MM:g} mm",
    )

### Distâncias baixas nas duas máscaras

In [ ]:
if pred_to_ref_mm is None or ref_to_pred_mm is None:
    print("Distâncias indisponíveis: pelo menos uma máscara está vazia.")
else:
    low_prediction = np.zeros_like(prediction_mask)
    low_reference = np.zeros_like(reference_mask)
    low_prediction[prediction_mask] = pred_to_ref_mm < HIGH_DISTANCE_MM
    low_reference[reference_mask] = ref_to_pred_mm < HIGH_DISTANCE_MM
    low_plot = plot_artery_masks(
        low_prediction,
        low_reference,
        scaled_spacing_mm,
        f"Distâncias baixas · < {HIGH_DISTANCE_MM:g} mm",
    )